# E-10b · Embedding de texto (paraphrase-multilingual-mpnet-base-v2)

Gera o embedding semantico de **titulo** e de **titulo + descricao + tags** para os 3.004 Shorts do Gold, para o experimento E-10b (protocolo pre-registrado em `docs/07`). So texto disponivel antes da publicacao.

**Entrada:** arraste `e10_text.csv` (pasta `case_galaxies/.tmp_transfer/`) para o painel Arquivos. **Saida:** `text_embeddings.npz` baixado automaticamente.

In [ ]:
import importlib, subprocess, sys, os, time
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
import numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer

MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
src = "/content/e10_text.csv"
assert os.path.exists(src), "arraste e10_text.csv para o painel Arquivos"
df = pd.read_csv(src).fillna("")
print(df.shape, "| GPU:", torch.cuda.is_available())

In [ ]:
def clean(text, limit):
    return " ".join(str(text).split())[:limit]

title = [clean(t, 300) for t in df["title"]]
title_desc = [clean(f"{t} | {d} | {g}", 1000) for t, d, g in zip(df["title"], df["description"], df["tags"])]
model = SentenceTransformer(MODEL, device="cuda" if torch.cuda.is_available() else "cpu")
t0 = time.time()
emb_title = model.encode(title, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
emb_title_desc = model.encode(title_desc, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
print(emb_title.shape, emb_title_desc.shape, f"{time.time() - t0:.0f}s")
np.savez("/content/text_embeddings.npz", video_id=df["video_id"].to_numpy(), title=emb_title.astype("float32"),
         title_desc=emb_title_desc.astype("float32"), source=MODEL, max_chars_title=300, max_chars_title_desc=1000)
print("ok /content/text_embeddings.npz")

In [ ]:
from google.colab import files
files.download("/content/text_embeddings.npz")